In [29]:
import librosa
import numpy as np
import os
from skimage.feature import local_binary_pattern


def extract_mfcc(y, sr, n_mfcc=80, n_fft=512, hop_length=160, win_length=400, n_mels=128):
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=n_mfcc,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        win_length=win_length
    )

    # if mfcc.shape[1] < 9: return None

    # delta = librosa.feature.delta(mfcc)

    # # delta-delta
    # delta2 = librosa.feature.delta(mfcc, order=2)

    # # ghép feature
    # feat_all = np.vstack([mfcc, delta, delta2]).T
    # CMVN
    return mfcc.T   # (num_frames, n_mfcc)


In [30]:
import numpy as np
import librosa
from skimage.feature import local_binary_pattern

def build_mfcc_dataset(
    wav_list,
    sr=16000,
    segment_sec=2,
    out_path="X_mfcc.npy"
):
    all_feats = []

    segment_len = int(segment_sec * sr)
    count = 0
    for wav_path in wav_list:
        count += 1
        print("Processing:", count )

        y, sr = librosa.load(wav_path, sr=sr)

        # nếu audio < 2s
        if len(y) < segment_len:
            segments = [y]
        else:
            num_segments = len(y) // segment_len
            segments = [
                y[i*segment_len:(i+1)*segment_len]
                for i in range(num_segments)
            ]

        for segment in segments:
            mfcc = extract_mfcc(segment, sr)
            #print(mfcc.shape)
            if mfcc is None: continue
            lbp = local_binary_pattern(
                mfcc,
                P=8,
                R=1,
                method="uniform"
            )

            hist, _ = np.histogram(
                lbp.ravel(),
                bins=256,
                range=(0, 256)
            )
            all_feats.append(hist)

        #print(f"{wav_path} -> {len(segments)} segments")

    X = np.vstack(all_feats)

    np.save(out_path, X)

    print("Saved:", out_path, X.shape)

    return X

In [31]:
import os
CLASS = "nonspeech_2"

DIR = r"E:\Pythonfile\Voice-Activity-Detect\data\processed\non-speech\non-mixed\music"
wav_list = [
    os.path.join(DIR, f)
    for f in os.listdir(DIR)
    if f.endswith(".wav")
]
print(wav_list)
X_mfcc = build_mfcc_dataset(
    wav_list,
    out_path=f"E:/Pythonfile/Voice-Activity-Detect/data/feature/train/mfcc_lbp_nonspeech_2_80.npy"
)

print(len(wav_list))
print(X_mfcc.shape)

['E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\non-speech\\non-mixed\\music\\1.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\non-speech\\non-mixed\\music\\10.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\non-speech\\non-mixed\\music\\100.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\non-speech\\non-mixed\\music\\101.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\non-speech\\non-mixed\\music\\102.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\non-speech\\non-mixed\\music\\103.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\non-speech\\non-mixed\\music\\104.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\non-speech\\non-mixed\\music\\105.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\non-speech\\non-mixed\\music\\106.wav', 'E:\\Pythonfile\\Voice-Activity-Detect\\data\\processed\\non-speech\\non-mixed\\music\\107.wav', 'E:\\Pythonfile\\Voice-Activity-